In [ ]:
!ls

orders.csv  restaurants.sql  sample_data  users.json


In [ ]:
!rm -r sample_data

In [ ]:
!ls

orders.csv  restaurants.sql  users.json


In [ ]:
import pandas as pd
import sqlite3
import json

In [ ]:
# Load orders.csv
orders = pd.read_csv("/content/orders.csv")  # Update path if different
orders.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [ ]:
# Load users.json
with open("/content/users.json") as f:
    users_data = json.load(f)

users = pd.json_normalize(users_data)
users.head()


,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [ ]:
# Create in-memory SQLite DB
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Read SQL file
with open("/content/restaurants.sql", 'r') as f:
    sql_script = f.read()

# Execute script to create tables and insert data
cursor.executescript(sql_script)

# Load restaurants table into pandas
restaurants = pd.read_sql_query("SELECT * FROM restaurants", conn)
restaurants.head()


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [ ]:
# Merge orders with users
df = orders.merge(users, on='user_id', how='left')

# Merge with restaurants
df = df.merge(restaurants, on='restaurant_id', how='left')

# Save final dataset
df.to_csv("/content/final_food_delivery_dataset.csv", index=False)
df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


City with highest revenue from Gold members

In [ ]:
gold_revenue = df[df['membership']=='Gold'].groupby('city')['total_amount'].sum()
gold_revenue.sort_values(ascending=False)


,total_amount
city,
Chennai,1080909.79
Pune,1003012.32
Bangalore,994702.59
Hyderabad,896740.19


Cuisine with highest average order value

In [ ]:
avg_order_by_cuisine = df.groupby('cuisine')['total_amount'].mean()
avg_order_by_cuisine.sort_values(ascending=False)


,total_amount
cuisine,
Mexican,808.021344
Italian,799.448578
Indian,798.466011
Chinese,798.389020


higgest avg order value across all orders

In [ ]:
# Group by cuisine and calculate average order value
avg_order_value = df.groupby('cuisine')['total_amount'].mean()

# Sort descending to see the highest
avg_order_value = avg_order_value.sort_values(ascending=False)

# Display results
print(avg_order_value)

# If you want just the cuisine with highest average order value
highest_avg_cuisine = avg_order_value.idxmax()
print(f"Cuisine with highest average order value: {highest_avg_cuisine}")


cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64
Cuisine with highest average order value: Mexican


Number of distinct users with total orders > 1000

In [ ]:
user_total = df.groupby('user_id')['total_amount'].sum()
(len(user_total[user_total>1000]))


2544

Restaurant rating range with highest revenue

In [ ]:
# See all column names in your final dataframe
print(df.columns)


Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name_x', 'name', 'city', 'membership', 'restaurant_name_y',
       'cuisine', 'rating'],
      dtype='object')


In [ ]:
# Make bins for ratings
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ['3.0-3.5','3.6-4.0','4.1-4.5','4.6-5.0']

df['rating_range'] = pd.cut(df['rating'], bins=bins, labels=labels, include_lowest=True)

# Group by rating range and sum revenue
rating_revenue = df.groupby('rating_range')['total_amount'].sum()
rating_revenue.sort_values(ascending=False)


/tmp/ipython-input-2431266043.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rating_revenue = df.groupby('rating_range')['total_amount'].sum()


,total_amount
rating_range,
4.6-5.0,2197030.75
3.0-3.5,2136772.70
4.1-4.5,1960326.26
3.6-4.0,1717494.41


Gold members – highest average order value city

In [ ]:
gold_avg_city = df[df['membership']=='Gold'].groupby('city')['total_amount'].mean()
gold_avg_city.sort_values(ascending=False)


,total_amount
city,
Chennai,808.459080
Hyderabad,806.421034
Bangalore,793.223756
Pune,781.162243


Cuisine with lowest number of distinct restaurants but high revenue

In [ ]:
cuisine_restaurants = df.groupby('cuisine')['restaurant_id'].nunique()
cuisine_revenue = df.groupby('cuisine')['total_amount'].sum()

# Combine
pd.DataFrame({'num_restaurants': cuisine_restaurants, 'revenue': cuisine_revenue}).sort_values(by=['num_restaurants', 'revenue'])


,num_restaurants,revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


% of orders placed by Gold members

In [ ]:
percentage_gold = (df[df['membership']=='Gold'].shape[0] / df.shape[0]) * 100
round(percentage_gold)


50

Restaurant with highest average order value but < 20 orders

In [ ]:
restaurant_stats = df.groupby('restaurant_name_y')['total_amount'].agg(['mean','count'])
restaurant_stats[restaurant_stats['count'] < 20].sort_values(by='mean', ascending=False)


,mean,count
restaurant_name_y,,
Restaurant_294,1040.222308,13
Restaurant_262,1029.473333,18
Restaurant_77,1029.180833,12
Restaurant_193,1026.306667,15
Restaurant_7,1002.140625,16
...,...,...
Restaurant_184,621.828947,19
Restaurant_498,596.815556,18
Restaurant_192,589.972857,14


In [ ]:


# Step 1: Rename restaurant_name_y to restaurant_name if not already done
df.rename(columns={'restaurant_name_y': 'restaurant_name'}, inplace=True)

# Step 2: Group by restaurant and calculate mean order value and count of orders
restaurant_stats = df.groupby('restaurant_name')['total_amount'].agg(['mean', 'count'])

# Step 3: Filter restaurants with less than 20 orders
small_restaurants = restaurant_stats[restaurant_stats['count'] < 20]

# Step 4: Sort by mean order value descending to get the highest
small_restaurants_sorted = small_restaurants.sort_values(by='mean', ascending=False)

# Step 5: Display top restaurant
print(small_restaurants_sorted.head(1))



                        mean  count
restaurant_name                    
Restaurant_294   1040.222308     13


Combination contributing highest revenue

In [ ]:
combo_revenue = df.groupby(['membership','cuisine'])['total_amount'].sum()
combo_revenue.sort_values(ascending=False)


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

Quarter with highest revenue

In [ ]:
# Convert order_date to datetime
df['order_date'] = pd.to_datetime(df['order_date'])

# Extract quarter
df['quarter'] = df['order_date'].dt.quarter

quarter_revenue = df.groupby('quarter')['total_amount'].sum()
quarter_revenue.sort_values(ascending=False)


/tmp/ipython-input-503261130.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['order_date'] = pd.to_datetime(df['order_date'])


,total_amount
quarter,
3,2037385.10
4,2018263.66
1,2010626.64
2,1945348.72


page 2

In [ ]:


# Step 0: Check your columns
print("Columns in your dataset:\n", df.columns, "\n")

# Step 1: Detect restaurant rating column automatically
rating_col = None
for col in df.columns:
    if 'rating' in col.lower():   # finds any column containing 'rating'
        rating_col = col
        break

if rating_col is None:
    raise ValueError("No restaurant rating column found! Check your dataset.")

print("Detected restaurant rating column:", rating_col, "\n")

 # Total orders by Gold members
total_gold_orders = df[df['membership'].str.lower()=='gold'].shape[0]

# Total revenue from Hyderabad orders (rounded)
hyderabad_revenue = round(df[df['city'].str.lower()=='hyderabad']['total_amount'].sum())

#  Number of distinct users who placed at least one order
distinct_users = df['user_id'].nunique()

#  Average order value for Gold members (2 decimals)
avg_gold_order = round(df[df['membership'].str.lower()=='gold']['total_amount'].mean(), 2)

# Orders for restaurants with rating >= 4.5
high_rating_orders = df[df[rating_col] >= 4.5].shape[0]

#  Orders in top revenue city among Gold members
gold_df = df[df['membership'].str.lower()=='gold']
top_gold_city = gold_df.groupby('city')['total_amount'].sum().idxmax()
orders_top_gold_city = gold_df[gold_df['city']==top_gold_city].shape[0]



print("  Total orders by Gold members:", total_gold_orders)
print("   Total revenue from Hyderabad (rounded):", hyderabad_revenue)
print("   Number of distinct users:", distinct_users)
print("  Average order value for Gold members:", avg_gold_order)
print(f"  Orders for restaurants with rating >= 4.5 ({rating_col}):", high_rating_orders)
print("   Orders in top revenue city among Gold members:", orders_top_gold_city)


Columns in your dataset:
 Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'name', 'city', 'membership', 'restaurant_name', 'cuisine', 'rating',
       'rating_range', 'quarter'],
      dtype='object') 

Detected restaurant rating column: rating 

  Total orders by Gold members: 4987
   Total revenue from Hyderabad (rounded): 1889367
   Number of distinct users: 2883
  Average order value for Gold members: 797.15
  Orders for restaurants with rating >= 4.5 (rating): 3374
   Orders in top revenue city among Gold members: 1337
